# Handwritten Formula to LaTeX OCR - Training and evaluating

This notebook is designed to run on Google Colab with GPU support.

**Requirements:**
- Google Colab with GPU runtime (T4 or better recommended)
- HuggingFace account (for model upload)

**Note:** TPU runtime is not recommended for this model. Use GPU (T4, A100) for better performance.

## 1. Setup

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    print(f"CUDA is available. GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
# Install dependencies
%pip install -q torch torchvision
%pip install -q transformers>=4.45.0 accelerate>=0.25.0
%pip install -q peft>=0.7.0 bitsandbytes>=0.41.0
%pip install -q datasets>=2.14.0
%pip install -q editdistance nltk sacrebleu
%pip install -q wandb python-dotenv

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)

In [ ]:
# Clone the repository
!git clone https://github.com/dmitryz1024/Multimodal_Reasoning_for_STEM.git
%cd Multimodal_Reasoning_for_STEM

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
import os

# Check if .env file exists
if not os.path.exists('.env'):
    print("Warning: .env file not found. Please create a .env file with your WANDB_API_KEY and HF_TOKEN")
    print("Example .env content:")
    print("WANDB_API_KEY=your_wandb_api_key_here")
    print("HF_TOKEN=your_huggingface_token_here")
else:
    print(".env file found")

# Load environment variables
load_dotenv()

# Login to Weights & Biases
import wandb
wandb_api_key = os.getenv("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)
    print("Successfully logged in to Weights & Biases")
else:
    print("Warning: WANDB_API_KEY not found in .env file. Wandb logging may not work.")

In [ ]:
# Verify setup
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA/GPU not available. Training will be slow on CPU.")
    print("Recommended: Change runtime to GPU (Runtime > Change runtime type > GPU)")

## 2. HuggingFace Login

In [ ]:
from huggingface_hub import login
import os

# Try to login with token from environment
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
    print("Logged in using HF_TOKEN environment variable")
else:
    print("Warning: HF_TOKEN not found in .env file.")
    print("Please set HF_TOKEN in your .env file or login manually using huggingface-cli")
    print("You can get a token from: https://huggingface.co/settings/tokens")

## 3. Load Datasets

In [ ]:
from datasets import load_dataset

# Load primary dataset
print("Loading LaTeX_OCR dataset...")
ds_latex_ocr = load_dataset("linxy/LaTeX_OCR", "human_handwrite")
print(ds_latex_ocr)

# Load secondary dataset (sample)
print("\nLoading MathWriting dataset...")
ds_mathwriting = load_dataset("deepcopy/MathWriting-human", split="train")
ds_mathwriting = ds_mathwriting.shuffle(seed=42).select(range(10000))  # Sample 10k
print(f"MathWriting samples: {len(ds_mathwriting)}")

## 4. Training Setup 1: LaTeX_OCR Only

In [ ]:
import sys
sys.path.insert(0, 'src')

from src.train import TrainConfig, train

# Configuration
# Note: 4-bit quantization is supported on GPU with bitsandbytes
config = TrainConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    primary_subset="human_handwrite",
    use_secondary=False,
    num_epochs=3,
    batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,  # 4-bit quantization for memory efficiency on GPU
    bf16=True,
)

print("Configuration:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")

# Verify device
import torch
print(f"\nDevice: GPU" if torch.cuda.is_available() else "Device: CPU")

In [ ]:
# Train with LaTeX_OCR only
model_latex_only, processor = train(
    config=config,
    run_name="sft_latex_ocr_only",
    wandb_project="handwritten-latex-ocr"
)

## 5. Training Setup 2: Combined Dataset

In [ ]:
# Update config for combined training
config.use_secondary = True
config.secondary_sample_size = 10000

# Train with combined dataset
model_combined, processor = train(
    config=config,
    run_name="sft_latex_ocr_mathwriting",
    wandb_project="handwritten-latex-ocr"
)

## 6. Evaluation

In [ ]:
from src.evaluate import run_full_evaluation, EvalConfig

eval_config = EvalConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    subset="human_handwrite",
    num_samples=70
)

results = run_full_evaluation(
    base_model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    checkpoint_latex_ocr="./checkpoints/sft_latex_ocr_only/final",
    checkpoint_combined="./checkpoints/sft_latex_ocr_mathwriting/final",
    config=eval_config
)

# Save results
import json
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to evaluation_results.json")

## 7. Upload to HuggingFace Hub

In [ ]:
# Upload LaTeX_OCR only model
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_only/final \
    --repo_id YOUR_USERNAME/latex-ocr-smolvlm-latex-only \
    --training_setup "SFT with LaTeX_OCR only"

In [ ]:
# Upload combined model
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_mathwriting/final \
    --repo_id YOUR_USERNAME/latex-ocr-smolvlm-combined \
    --training_setup "SFT with LaTeX_OCR + MathWriting"

## 8. Test Inference

In [ ]:
from src.inference import LatexOCRInference
from PIL import Image
import matplotlib.pyplot as plt

# Load fine-tuned model
engine = LatexOCRInference(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    adapter_path="./checkpoints/sft_latex_ocr_only/final"
)

# Test on a few examples
test_ds = ds_latex_ocr['test']

for i in range(5):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = engine.predict(image)
    
    print(f"\nExample {i+1}:")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    
    plt.figure(figsize=(6, 2))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

## 9. Download Results

In [ ]:
# Download evaluation results and checkpoints
from google.colab import files

files.download('evaluation_results.json')

# Optionally zip and download checkpoints
!zip -r checkpoints.zip checkpoints/
files.download('checkpoints.zip')